In [1]:
import sys
sys.path.append("..")  # utilsは1階層上にある

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from utils.models import LSTMGaussian,PortfolioLSTMModel
from utils.trainer import DreamingTrainer
from utils.datawindow import TorchDataWindow
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

In [2]:
import pandas as pd

# データ読み込み
df = pd.read_csv('../data/100stock_data.csv')
# date列を削除
df = df.drop(columns=['date'])
# 対数収益
prices = df.astype(float).values
logret = np.diff(np.log(prices), axis=0)        # [L-1, 100]
cols = list(df.columns)

target_col = 0
y   = logret[:, [target_col]]                   # ← ここを [] で2次元に（[L-1,1]）
exo = np.delete(logret, target_col, axis=1)     # [L-1, 99]

seq_single = np.concatenate([y, exo], axis=1)   # [L-1, 100]
data_single = [seq_single.astype(np.float32)]
D = data_single[0].shape[1]

data_all = []
for c in range(logret.shape[1]):
    y_c   = logret[:, [c]]
    exo_c = np.delete(logret, c, axis=1)
    seq_c = np.concatenate([y_c, exo_c], axis=1).astype(np.float32)  # [L-1, 100]
    data_all.append(seq_c)
# 使うモードを選ぶ：
use_all_targets = False
data = data_all if use_all_targets else data_single
D    = data[0].shape[1]

#データ分割
# logret: [T, N]  (T=時点数, N=銘柄数)
test_len = 200

def split_train_test_seq(seq_2d, test_len):
    # seq_2d: [L, D]（列0=ターゲット, 列1..=外生）
    L = seq_2d.shape[0]
    assert test_len < L
    return seq_2d[:L-test_len], seq_2d[L-test_len:]

# 例：単一ターゲット＋外生（D=100）
seq = seq_single.astype(np.float32)         # [L, D]
train_seq, test_seq = split_train_test_seq(seq, test_len)

# 複数系列（全銘柄を個別に学習したい場合）
train_data = []
test_data  = []
for seq in data_all:                        # 各seq: [L, D]
    tr, te = split_train_test_seq(seq, test_len)
    train_data.append(tr); test_data.append(te)

In [3]:
# 最適パラメータの読み込み
import json
with open("../results/best_params.json", "r") as f:
    best = ("best", json.load(f))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMGaussian(input_dim=D, hidden_size=best[1]["hidden"],num_layers=best[1]["layers"],
                     proj_dim=best[1]["proj_dim"], dropout=best[1]["dropout"]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=best[1]["lr"])

trainer = DreamingTrainer(
    model=model, optimizer=optimizer, device=device,
    interleave_mode=best[1]["mode"], K_interleave=best[1]["K"],
    epochs=500,
    warm_vanilla_steps=500,
    max_vanilla_steps_per_epoch=200,
    dreaming_steps_per_epoch=50,
    dreaming_seq_len=50,
    lambda_d=0.2,
    temperature=best[1]["T"],
    temperature_schedule="constant",
    temperature_params=None,
    exo_mode="hold",
    grad_clip=1.0,
    logvar_clamp=(-20, 10),
    seed=42
)
trainer.train(train_data, val_data=None)

[WARMUP] vanilla_steps=500
[WARMUP] step 50, loss=-2.540979, mse=0.000514, mae=0.016357
[INTERLEAVE] (batch mode) K=8, T=1.800
[BATCH MODE] VANILLA avg_loss=-3.309281, mse=0.000391, mae=0.013251
[BATCH MODE] DREAMING avg_loss=-2.603627, mse=0.000709, mae=0.021181
[E1] SUMMARY: vanilla_loss=-3.309281, dreaming_loss=-2.603627, T=1.800
[INTERLEAVE] (batch mode) K=8, T=1.800
[BATCH MODE] VANILLA avg_loss=-3.355567, mse=0.000380, mae=0.013172
[BATCH MODE] DREAMING avg_loss=-2.796220, mse=0.000593, mae=0.019126
[E2] SUMMARY: vanilla_loss=-3.355567, dreaming_loss=-2.796220, T=1.800
[INTERLEAVE] (batch mode) K=8, T=1.800
[BATCH MODE] VANILLA avg_loss=-3.351543, mse=0.000375, mae=0.013138
[BATCH MODE] DREAMING avg_loss=-2.472455, mse=0.003575, mae=0.032491
[E3] SUMMARY: vanilla_loss=-3.351543, dreaming_loss=-2.472455, T=1.800
[INTERLEAVE] (batch mode) K=8, T=1.800
[BATCH MODE] VANILLA avg_loss=-3.355566, mse=0.000371, mae=0.013110
[BATCH MODE] DREAMING avg_loss=-2.687301, mse=0.018762, mae=0.04

In [4]:
train_metrics = trainer.evaluate_metrics(train_data)
print(f"Train Data NLL: {train_metrics['nll']:.6f}")
print(f"Train Data MSE: {train_metrics['mse']:.6f}")
print(f"Train Data MAE: {train_metrics['mae']:.6f}")

Train Data NLL: -3.888911
Train Data MSE: 0.000210
Train Data MAE: 0.009905


In [5]:
#学習済みモデルをmodelsフォルダに保存
import os
#os.makedirs("../models", exist_ok=True)
model_path = "../models/dreaming_model.pth"
torch.save(model.state_dict(), model_path)